Purpose of this notebook: to take apart video feed and turn each frame into an image that then can be run in MegaDetector, in batches.

Designed to run on JupyterHub / Linux / macOS / Windows
NOTE! this version of megadetector does not import a module, you install from a git clone into your directory and env with python 3.10, no higher.
install the requirements.txt inside the kernel/env you will create. ideas- frame rate function to detect? what files to keep? where? naming? use OOP? permissions for
users? how to manage files? recursive through subfolders? delete option for jsons and/or jpegs? what is workflow? what does timelapse need as input?
note if using chatgpt-must must stipulate which megadetector you are using, mixing pytorchwildlife and dan morris' would be disaster code. cuda and GPU being used?
how long does it take on a tower? Jetstream2? CPU only machine? test the same video on each system, run %timeit and compare.

make sure you have run conda activate megadetector (the env NOT dan morris' if you need to install. also make sure you are in megadetector kernel
also you might need to install megadetector like this, if you did the git clone method, example: cd /home/jupyter-bernie/MegaDetector
pip install -e .


Note: this has not been battle-tested for how many videos it can process!

In [2]:
# Dan Morris' Megadetector version
# # MegaDetector v5 – video → frames → detection → video reconstruction  
# This notebook is now **platform independent** (Linux / macOS / Windows).
import os
import sys
import json
import warnings
import subprocess
import shutil
from pathlib import Path


import cv2                # opencv‑python‑headless works on all OSes
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output


import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Using device: {DEVICE}")

🖥️  Using device: cuda


In [ ]:
#Example Root of your (from Dan Morris) MegaDetector repo
MEGADETECTOR_ROOT = Path("/home/jupyter-bernie/MegaDetector")

#Example Folder containing videos (can have nested subfolders)
DATA_DIR = Path("/home/jupyter-bernie/6June2025")

# Output location (mirrors input structure)
OUTPUT_DIR = Path("./megadetector_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model name (official MegaDetector models)
MODEL_NAME = "MDV5A"

# Frames per second for stitched video
OUTPUT_FPS = 30  # iPhone / camera trap safe default

# MegaDetector animal category ID (INT, not string)
ANIMAL_CATEGORY_ID = 1

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}

print("Configuration:")
print(f"MegaDetector root: {MEGADETECTOR_ROOT}")
print(f"Input video folder: {DATA_DIR}")
print(f"Output folder: {OUTPUT_DIR}")
print(f"Model: {MODEL_NAME}")


Configuration:
MegaDetector root: /home/jupyter-bernie/MegaDetector
Input video folder: /home/jupyter-bernie/6June2025
Output folder: megadetector_outputs
Model: MDV5A


In [4]:
videos = [
    p for p in DATA_DIR.rglob("*")
    if p.suffix.lower() in VIDEO_EXTS
]

print(f"Found {len(videos)} videos")
videos[:5] #change the 5 to the number of videos if you would like to see them all and rerun cell


Found 20 videos


[PosixPath('/home/jupyter-bernie/6June2025/Location_2/DCIM/101EK113/IMG_0007.AVI'),
 PosixPath('/home/jupyter-bernie/6June2025/Location_2/DCIM/101EK113/IMG_0133.MP4'),
 PosixPath('/home/jupyter-bernie/6June2025/Location_2/DCIM/101EK113/05110004.AVI'),
 PosixPath('/home/jupyter-bernie/6June2025/Location_2/DCIM/101EK113/IMG_0011.AVI'),
 PosixPath('/home/jupyter-bernie/6June2025/Location_2/DCIM/101EK113/06070036.AVI')]

In [4]:
# sanity check 
assert MEGADETECTOR_ROOT.exists(), "MegaDetector repo not found"
assert DATA_DIR.exists(), "Input data directory not found"

run_detector_script = MEGADETECTOR_ROOT / "megadetector/detection/run_detector_batch.py"
assert run_detector_script.exists(), "run_detector_batch.py not found"

print("✔ All required paths exist")


✔ All required paths exist


In [5]:
# Helper: find all videos recursively- important to make sure this is finding all videos in subfolders!
def find_all_videos(root_dir):
    VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv"}

    videos = []
    for root, _, files in os.walk(root_dir):
        root = Path(root)

        # Skip output directories
        if "megadetector_outputs" in root.parts:
            continue

        for f in files:
            p = Path(f)
            if (
                p.suffix.lower() in VIDEO_EXTS
                and "_detected" not in p.stem
            ):
                videos.append(root / p)

    return sorted(videos)



In [6]:
# this really checks if you found all videos in subfolders or anywhere by calling function above
videos = find_all_videos(DATA_DIR)

print(f"Found {len(videos)} videos")
for v in videos[:10]:
    print(" ", v.relative_to(DATA_DIR))


Found 22 videos
  Location_1/DCIM/100EK113/IMG_0004.AVI
  Location_1/DCIM/100EK113/IMG_0041.AVI
  Location_1/DCIM/100EK113/IMG_0042.AVI
  Location_1/DCIM/100EK113/IMG_0058.AVI
  Location_1/DCIM/100EK113/IMG_0258.AVI
  Location_1/DCIM/101EK113/IMG_0011.AVI
  Location_1/DCIM/101EK113/IMG_0012.AVI
  Location_1/DCIM/101EK113/IMG_0014.AVI
  Location_1/DCIM/101EK113/IMG_0022.AVI
  Location_1/DCIM/101EK113/IMG_0047.AVI


In [7]:
#checking what extensions exist in DATA_DIR- troubleshoots above if having issues
from collections import Counter

ext_counts = Counter()

for root, _, files in os.walk(DATA_DIR):
    for f in files:
        ext_counts[Path(f).suffix.lower()] += 1

ext_counts


Counter({'.jpg': 1080, '.avi': 20, '.mp4': 5, '.json': 2, '.mov': 1})

In [8]:
# Given a video path, where do its frames, JSON and output video live?
def video_output_dirs(video_path):
    """
    Given a video path under DATA_DIR, return all output paths
    """
    rel = video_path.relative_to(DATA_DIR)
    base = OUTPUT_DIR / rel.with_suffix("")

    frames_dir = base / "frames"
    json_path = base / "detections.json"
    output_video = base.with_name(base.name + "_detected.mp4")

    frames_dir.mkdir(parents=True, exist_ok=True)
    base.mkdir(parents=True, exist_ok=True)

    return frames_dir, json_path, output_video


In [9]:
#troubleshooting issues
for video_path in videos:
    frames_dir, output_json, output_video = video_output_dirs(video_path)

    print("frames_dir =", frames_dir)
    print("OUTPUT_DIR =", OUTPUT_DIR)

    break  # just inspect the first one


frames_dir = megadetector_outputs/Location_1/DCIM/100EK113/IMG_0004/frames
OUTPUT_DIR = megadetector_outputs


In [10]:
#helper clean frames directory safely
def cleanup_extractions(frames_dir):
    frames_dir = Path(frames_dir)

    # Safety checks
    assert frames_dir.name == "frames", f"Refusing to delete non-frames dir: {frames_dir}"
    assert OUTPUT_DIR in frames_dir.parents, f"{frames_dir} is not inside OUTPUT_DIR"

    if frames_dir.exists():
        shutil.rmtree(frames_dir)

    frames_dir.mkdir(parents=True, exist_ok=True)


In [17]:
def resolve_frame_path(frames_dir, file_field):
    p = Path(file_field)

    # If MegaDetector already stored a path, use it
    if p.is_absolute() or p.exists():
        return p

    # Otherwise, assume it's relative to frames_dir
    return frames_dir / p


In [11]:
def extract_frames(video_path, frames_dir):
    frames_dir = Path(frames_dir)
    frames_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        "ffmpeg",
        "-y",                    # overwrite if exists
        "-i", str(video_path),
        "-q:v", "2",             # good JPEG quality
        str(frames_dir / "frame_%06d.jpg")
    ]

    subprocess.run(cmd, check=True)


In [12]:
# helper check if frame has animal detections  HERE IS WHERE CONF THRESH IS SET TO .5
def has_animal_detection(frame_info, conf_thresh=0.5):
    for d in frame_info:
        if (
            d.get("category") == ANIMAL_CATEGORY_ID
            and d.get("confidence", 0) >= conf_thresh
        ):
            return True
    return False


In [13]:
# helper draw bounding boxes (animal only)
def draw_boxes(frame, detections, conf_thresh=0.8):
    h, w = frame.shape[:2]

    for det in detections:
        if (
            det.get("category") != ANIMAL_CATEGORY_ID
            or det.get("confidence", 0) < conf_thresh
        ):
            continue

        x, y, bw, bh = det["bbox"]
        x1 = int(x * w)
        y1 = int(y * h)
        x2 = int((x + bw) * w)
        y2 = int((y + bh) * h)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            frame,
            f"animal {det['confidence']:.2f}",
            (x1, max(y1 - 5, 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            1,
        )

    return frame


In [14]:
# helper stitch only frames with detections into video
def create_detected_video_only(md_json_path, frames_dir, output_video_path, conf_thresh=0.5):
    with open(md_json_path, "r") as f:
        md = json.load(f)

    images = md.get("images", [])

    # ---- Phase 1: global check ----
    has_any_animals = any(
        has_animal_detection(img.get("detections", []), conf_thresh)
        for img in images
    )

    if not has_any_animals:
        print("No animal detections found in entire video, skipping.")
        return

    # ---- Phase 2: initialize video writer using first readable frame ----
    first_frame = None
    first_path = None

    for img in images:
        p = frames_dir / img["file"]
        frame = cv2.imread(str(p))
        if frame is not None:
            first_frame = frame
            first_path = p
            break

    if first_frame is None:
        raise RuntimeError("No readable frames found to initialize video.")

    h, w = first_frame.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(str(output_video_path), fourcc, OUTPUT_FPS, (w, h))

    written = 0

    # ---- Phase 3: write ALL frames ----
    for img in images:
        frame_path = frames_dir / img["file"]
        frame = cv2.imread(str(frame_path))
        if frame is None:
            continue

        detections = img.get("detections", [])

        # Draw boxes ONLY if animal detections exist
        if has_animal_detection(detections, conf_thresh):
            frame = draw_boxes(frame, detections, conf_thresh)

        out.write(frame)
        written += 1

    out.release()
    print(f"🎬 Wrote full video ({written} frames) to {output_video_path}")



In [18]:
all_videos = find_all_videos(DATA_DIR)
print(f"Found {len(all_videos)} videos")

for video_path in all_videos:
    print("\n==============================")
    print("Processing:", video_path)

    #  ALWAYS derive paths this way
    frames_dir, output_json, output_video = video_output_dirs(video_path)
    #TESTS
    print("frames_dir:", frames_dir)
    print("output_json:", output_json)
    print("output_video:", output_video)
    # HARD RESET 
    cleanup_extractions(frames_dir)

    #EXTRACT
    extract_frames(video_path, frames_dir)

    # ASSERT: frames were actually extracted
    frame_files = sorted(frames_dir.glob("frame_*.jpg"))
    assert len(frame_files) > 0, f"No frames extracted for {video_path}"

    print(f"Extracted {len(frame_files)} frames")

    # --- Detect FPS ---
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    print(f"FPS: {fps}")

    # --- Extract frames ---
    cap = cv2.VideoCapture(str(video_path))
    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_file = frames_dir / f"frame_{frame_count:05d}.jpg"
        cv2.imwrite(str(frame_file), frame)
        frame_count += 1
    cap.release()
    print(f"Extracted {frame_count} frames")

    # --- Run MegaDetector ---
    !python /home/jupyter-bernie/MegaDetector/megadetector/detection/run_detector_batch.py \
        {MODEL_NAME} \
        {frames_dir} \
        {output_json}

    # --- Load JSON ---
    with open(output_json) as f:
        md_json = json.load(f)

    image_detections = md_json.get("images", [])
    if not image_detections:
        print("⚠️ No detections, skipping video")
        cleanup_extractions(frames_dir)
        continue

    # --- Initialize video writer ---
    
    first_frame_path = resolve_frame_path(frames_dir, image_detections[0]["file"])
    first_frame = cv2.imread(str(first_frame_path))

    if first_frame is None:
        raise RuntimeError(f"Failed to read first frame: {first_frame_path}")

    height, width, _ = first_frame.shape

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(str(output_video), fourcc, fps, (width, height))

    # --- Write FULL video (boxes only on animal frames) ---
    written = 0
    for frame_info in image_detections:
        frame_path = resolve_frame_path(frames_dir, frame_info["file"])
        frame = cv2.imread(str(frame_path))
        if frame is None:
            continue


        if has_animal_detection(frame_info.get("detections", [])):
            frame = draw_boxes(frame, frame_info["detections"])

        out.write(frame)
        written += 1

    out.release()
    print(f"✅ Wrote {written} frames to {output_video}")

    # --- Cleanup ---
    cleanup_extractions(frames_dir)


Found 22 videos

Processing: /home/jupyter-bernie/6June2025/Location_1/DCIM/100EK113/IMG_0004.AVI
FPS: 30.00030000300003
Extracted 307 frames
Model v5a.0.1 already exists and is valid at /tmp/megadetector_models/md_v5a.0.1.pt
307 image files found in the input directory
PyTorch reports 2 available CUDA devices
GPU available: True
PyTorch reports 2 available CUDA devices
GPU available: True
Loading PT detector with compatibility mode classic
Loaded image size 1280 from model metadata
Using model stride: 64
PTDetector using device cuda:0
Fusing layers... 
Fusing layers... 
Model summary: 733 layers, 140054656 parameters, 0 gradients, 208.8 GFLOPs
Model summary: 733 layers, 140054656 parameters, 0 gradients, 208.8 GFLOPs
Loaded model in 2.53 seconds
100%|█████████████████████████████████████████| 307/307 [00:07<00:00, 40.11it/s]
Finished inference for 307 images in 11.07 seconds (27.74 images per second)
Output file saved at megadetector_outputs/Location_1/DCIM/100EK113/IMG_0004/detection

In [ ]:

video_path = videos[0]  # debug with one video

frames_dir, output_json, output_video = video_output_dirs(video_path)

cleanup_extractions(frames_dir)

extract_frames(video_path, frames_dir)

run_megadetector(frames_dir, output_json)

create_detected_video_only(
    output_json,
    frames_dir,
    output_video
)

cleanup_extractions(frames_dir)


In [20]:
with open(output_json) as f:
    md = json.load(f)

for img in md["images"]:
    if img.get("detections"):
        print(img["detections"][0])
        break


{'category': '1', 'conf': 0.885, 'bbox': [0.6945, 0.5097, 0.1656, 0.1347]}
